In [ ]:
# ==========================================================
# MOUNT GOOGLE DRIVE
# ==========================================================
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip -q install xgboost lightgbm shap joblib catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.5 MB/s eta 0:00:00


In [ ]:
# ==========================================================
# IMPORT LIBRARIES
# ==========================================================

import warnings
warnings.filterwarnings("ignore")

import os
import time
import joblib
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt


from sklearn.model_selection import (
    train_test_split,
    RandomizedSearchCV,
    KFold
)

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error,
    mean_absolute_percentage_error
)

from sklearn.ensemble import RandomForestRegressor

from sklearn.svm import SVR

from xgboost import XGBRegressor

from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

from scipy.stats import randint
from scipy.stats import uniform

In [ ]:
# ==========================================================
# PROJECT DIRECTORIES
# ==========================================================

PROJECT_DIR = "/content/drive/MyDrive/Biomass_Project"

DATA_DIR = os.path.join(PROJECT_DIR, "01_Data")
MODEL_DIR = os.path.join(PROJECT_DIR, "02_Models")
PRED_DIR = os.path.join(PROJECT_DIR, "03_Predictions")
CV_DIR = os.path.join(PROJECT_DIR, "04_CV_Results")
TABLE_DIR = os.path.join(PROJECT_DIR, "10_Tables")

for folder in [
    MODEL_DIR,
    PRED_DIR,
    CV_DIR,
    TABLE_DIR
]:
    os.makedirs(folder, exist_ok=True)

print("Folders Ready")

Folders Ready


In [ ]:
# ==========================================================
# LOAD DATASET
# ==========================================================

csv_path = os.path.join(
    DATA_DIR,
    "Bangladesh_AGBD_RH95_Predictors_Final_Clean.csv"
)

df = pd.read_csv(csv_path)

print(df.shape)

df.head()

(1227, 17)


,system:index,EVI,Elevation,NBR,NDMI,NDVI,Slope,VH,VV,agbd,rh95,VV_minus_VH,canopy_relief,Canopy_Cover,FHD,forest_name,PAI
0,0_0,0.446794,67,0.521174,0.262418,0.712379,5,-13.651795,-7.526714,133.741379,20.219999,6.125081,0.465772,0.811137,3.103661,Lawachara,3.337479
1,1_0,0.471140,72,0.589634,0.302198,0.784394,9,-13.769139,-5.908260,82.173004,25.500000,7.860879,0.695833,0.513298,2.886727,Lawachara,1.440718
2,2_0,0.521434,75,0.603416,0.335680,0.779537,3,-15.444711,-7.497519,82.173004,14.820000,7.947192,0.695833,0.513298,2.886727,Lawachara,1.440718
3,3_0,0.449244,71,0.593663,0.314374,0.757451,6,-13.838805,-7.893732,189.087280,32.540001,5.945073,0.541433,0.803823,3.229514,Lawachara,3.261589
4,4_0,0.359919,74,0.591196,0.313946,0.740298,1,-14.722586,-6.831360,79.405884,24.000000,7.891227,0.402498,0.825644,2.649440,Lawachara,3.497878


In [ ]:
# ==========================================================
# REPRESENTATIVE FEATURE LISTS
# ==========================================================

# ----------------------------------------------------------
# GEDI Structural Features
# ----------------------------------------------------------

gedi_features = [

    'rh95', 'Canopy_Cover', 'PAI',

]




# ----------------------------------------------------------
# Vegetation Indices
# ----------------------------------------------------------

veg_features = [

    'NDVI','EVI',
    'NDMI','NBR',

]


# ----------------------------------------------------------
# Sentinel-1 Backscatter
# ----------------------------------------------------------

sar_features = [

    'VV',
    'VH',
    'VV_minus_VH',

]


# ----------------------------------------------------------
# Terrain
# ----------------------------------------------------------

terrain_features = [

    'Elevation',
    'Slope'

]

In [ ]:
feature_sets = {

    "GEDI": (
        gedi_features
    ),

    "GEDI_Optical": (
        gedi_features +
        veg_features
    ),

    "Full_Feature_Set": (
        gedi_features +
        veg_features +
        sar_features +
        terrain_features
    ),

}
target = "agbd"

In [ ]:
# ==========================================================
# TRAIN TEST SPLIT
# ==========================================================

train_df, test_df = train_test_split(

    df,

    test_size=0.30,

    random_state=42

)

print(train_df.shape)

print(test_df.shape)

(858, 17)
(369, 17)


In [ ]:
# ==========================================================
# INNER CV
# ==========================================================

inner_cv = KFold(

    n_splits=5,

    shuffle=True,

    random_state=42

)

In [ ]:
models = {

"CatBoost":
Pipeline([
    ("model",
     CatBoostRegressor(random_state=42, verbose=0))
]),
"RF":

Pipeline([

("model",
RandomForestRegressor(random_state=42))

]),

"SVR":

Pipeline([

("scaler",StandardScaler()),

("model",SVR())

]),

"XGB":

Pipeline([

("model",

XGBRegressor(

random_state=42,

objective="reg:squarederror",

verbosity=0

))

]),

"LGBM":

Pipeline([

("model",

LGBMRegressor(

random_state=42,

verbose=-1

))

])
}

In [ ]:
# ==========================================================
# HYPERPARAMETER SEARCH SPACES
# ==========================================================

param_grids = {
    "CatBoost": {

        "model__iterations": randint(100, 401),

        "model__depth": randint(4,9),
        "model__learning_rate":[
            0.03,
            0.05,
            0.1
        ],
        "model__l2_leaf_reg": randint(1,8),
        "model__subsample":[
            0.8,
            1.0
        ],
        "model__random_strength": randint(1,6)
    },

    "RF": {

        "model__n_estimators": randint(50, 701),

        "model__max_depth": [None] + list(range(2, 21)),

        "model__min_samples_split": randint(2, 11),

        "model__min_samples_leaf": randint(1, 6),

        "model__max_features": [
            "sqrt",
            "log2",
            None
        ]

    },

    "SVR": {

        "model__C": [
            0.1,1,2,3,4,5,
            10,20,30,40,50
        ],

        "model__epsilon": [
            0.001,
            0.01,
            0.1
        ],

        "model__kernel": [
            "linear",
            "rbf",
            "poly",
            "sigmoid"
        ],

        "model__gamma": [
            "scale",
            "auto"
        ]

    },

    "XGB": {

        "model__n_estimators": randint(50,701),

        "model__max_depth": randint(3,11),

        "model__learning_rate": [
            0.01,
            0.05,
            0.1,
            0.2
        ],

        "model__subsample":[
            0.6,
            0.8,
            1.0
        ],

        "model__colsample_bytree":[
            0.6,
            0.8,
            1.0
        ],

        "model__reg_alpha":[
            0,
            0.5,
            1.0
        ],

        "model__reg_lambda":[
            0,
            0.5,
            1.0
        ]

    },

    "LGBM": {

        "model__n_estimators": randint(50,701),

        "model__learning_rate":[
            0.01,
            0.05,
            0.1,
            0.2
        ],

        "model__max_depth":[
            -1,
            3,4,5,6,7,8,9,10,11,12,13,14,15
        ],

        "model__num_leaves": randint(20,151),

        "model__subsample":[
            0.6,
            0.8,
            1.0
        ],

        "model__colsample_bytree":[
            0.6,
            0.8,
            1.0
        ],

        "model__min_child_samples": randint(5,51),

        "model__reg_alpha":[
            0,
            0.5,
            1.0
        ],

        "model__reg_lambda":[
            0,
            0.5,
            1.0
        ]

    }

}

In [ ]:
# ==========================================================
# MASTER STORAGE
# ==========================================================

experiment_results = {}

summary_results = []

In [ ]:
print("Total NaNs in df:", df.isna().sum().sum())

na_cols = df.isna().sum()
print(na_cols[na_cols > 0].sort_values(ascending=False))

Total NaNs in df: 0
Series([], dtype: int64)


In [ ]:
# ==========================================================
# MAIN TRAINING LOOP
# ==========================================================

for feature_name, feature_columns in feature_sets.items():

    print("\n")
    print("="*90)
    print(f"FEATURE SET : {feature_name}")
    print("="*90)

    # -------------------------------
    # Select Features
    # -------------------------------

    X_train = train_df[feature_columns]
    X_test = test_df[feature_columns]

    y_train = train_df[target]
    y_test = test_df[target]

    experiment_results[feature_name] = {}

    # -------------------------------
    # Loop through models
    # -------------------------------

    for model_name, pipeline in models.items():

        print("\n")
        print("-"*70)
        print(f"Training {model_name}")
        print("-"*70)

        start_train = time.time()

        search = RandomizedSearchCV(

            estimator=pipeline,

            param_distributions=param_grids[model_name],

            n_iter=50,

            scoring="r2",

            cv=inner_cv,

            random_state=42,

            n_jobs=-1,

            verbose=1,

            return_train_score=True

        )

        search.fit(

            X_train,

            y_train

        )

        end_train = time.time()

        training_time = end_train - start_train

        best_model = search.best_estimator_

        # -------------------------------
        # Prediction
        # -------------------------------

        start_pred = time.time()

        prediction = best_model.predict(X_test)

        end_pred = time.time()

        prediction_time = end_pred - start_pred

        # -------------------------------
        # Metrics
        # -------------------------------

        r2 = r2_score(

            y_test,

            prediction

        )

        rmse = np.sqrt(

            mean_squared_error(

                y_test,

                prediction

            )

        )

        mae = mean_absolute_error(

            y_test,

            prediction

        )

        mape = mean_absolute_percentage_error(

            y_test,

            prediction

        )

        rrmse = (

            rmse /

            y_test.mean()

        ) * 100

        # -------------------------------
        # Prediction dataframe
        # -------------------------------

        prediction_df = pd.DataFrame({

            "Observed":

            y_test.values,

            "Predicted":

            prediction

        })

        # -------------------------------
        # Store Everything
        # -------------------------------

        experiment_results[feature_name][model_name] = {

            "best_model":

            best_model,

            "search_object":

            search,

            "predictions":

            prediction_df,

            "cv_results":

            pd.DataFrame(

                search.cv_results_

            ),

            "metrics": {

                "Best CV R2":

                search.best_score_,

                "Test R2":

                r2,

                "RMSE":

                rmse,

                "MAE":

                mae,

                "MAPE":

                mape,

                "rRMSE (%)":

                rrmse,

                "Training Time (sec)":

                training_time,

                "Prediction Time (sec)":

                prediction_time

            },

            "best_parameters":

            search.best_params_

        }

        # -------------------------------
        # Summary Table
        # -------------------------------

        summary_results.append({

            "Feature Set":

            feature_name,

            "Model":

            model_name,

            "Best CV R2":

            search.best_score_,

            "Test R2":

            r2,

            "RMSE":

            rmse,

            "MAE":

            mae,

            "MAPE":

            mape,

            "rRMSE (%)":

            rrmse,

            "Training Time (sec)":

            training_time,

            "Prediction Time (sec)":

            prediction_time

        })

        print(f"Finished : {model_name}")

        print(f"Best CV R² : {search.best_score_:.4f}")

        print(f"Test R²    : {r2:.4f}")

        print(f"RMSE       : {rmse:.4f}")

        print(f"MAE        : {mae:.4f}")



FEATURE SET : GEDI


----------------------------------------------------------------------
Training CatBoost
----------------------------------------------------------------------
Fitting 5 folds for each of 50 candidates, totalling 250 fits
Finished : CatBoost
Best CV R² : 0.6797
Test R²    : 0.6762
RMSE       : 34.2189
MAE        : 22.0112


----------------------------------------------------------------------
Training RF
----------------------------------------------------------------------
Fitting 5 folds for each of 50 candidates, totalling 250 fits
Finished : RF
Best CV R² : 0.6683
Test R²    : 0.6602
RMSE       : 35.0577
MAE        : 21.9826


----------------------------------------------------------------------
Training SVR
----------------------------------------------------------------------
Fitting 5 folds for each of 50 candidates, totalling 250 fits
Finished : SVR
Best CV R² : 0.6639
Test R²    : 0.6536
RMSE       : 35.3937
MAE        : 19.4058


---------------------

In [ ]:
# ==========================================================
# SAVE ALL RESULTS
# ==========================================================

print("=" * 80)
print("Saving models and results...")
print("=" * 80)

for feature_name in experiment_results:

    # Create folders for each feature set
    feature_model_dir = os.path.join(MODEL_DIR, feature_name)
    feature_pred_dir = os.path.join(PRED_DIR, feature_name)
    feature_cv_dir = os.path.join(CV_DIR, feature_name)

    os.makedirs(feature_model_dir, exist_ok=True)
    os.makedirs(feature_pred_dir, exist_ok=True)
    os.makedirs(feature_cv_dir, exist_ok=True)

    for model_name in experiment_results[feature_name]:

        result = experiment_results[feature_name][model_name]

        # --------------------------------------------------
        # Save trained model
        # --------------------------------------------------

        model_path = os.path.join(
            feature_model_dir,
            f"{model_name}.pkl"
        )

        joblib.dump(
            result["best_model"],
            model_path
        )

        # --------------------------------------------------
        # Save predictions
        # --------------------------------------------------

        prediction_path = os.path.join(
            feature_pred_dir,
            f"{model_name}_Predictions.csv"
        )

        result["predictions"].to_csv(
            prediction_path,
            index=False
        )

        # --------------------------------------------------
        # Save CV Results
        # --------------------------------------------------

        cv_path = os.path.join(
            feature_cv_dir,
            f"{model_name}_CV_Results.csv"
        )

        result["cv_results"].to_csv(
            cv_path,
            index=False
        )

        # --------------------------------------------------
        # Save Best Parameters
        # --------------------------------------------------

        parameter_path = os.path.join(
            feature_cv_dir,
            f"{model_name}_Best_Parameters.txt"
        )

        with open(parameter_path, "w") as f:

            for key, value in result["best_parameters"].items():

                f.write(f"{key} : {value}\n")

print("Everything Saved Successfully.")

Saving models and results...
Everything Saved Successfully.


In [ ]:
# ==========================================================
# SUMMARY TABLE
# ==========================================================

results_df = pd.DataFrame(summary_results)

results_df = results_df.sort_values(
    by="Test R2",
    ascending=False
)

results_df.reset_index(
    drop=True,
    inplace=True
)

display(results_df)

,Feature Set,Model,Best CV R2,Test R2,RMSE,MAE,MAPE,rRMSE (%),Training Time (sec),Prediction Time (sec)
0,Full_Feature_Set,RF,0.703216,0.731663,31.152472,18.717204,0.181380,26.538594,431.852057,0.056931
1,Full_Feature_Set,SVR,0.677607,0.729638,31.269785,19.154242,0.200700,26.638533,22.854115,0.048022
2,Full_Feature_Set,CatBoost,0.711262,0.724459,31.567871,19.753955,0.210147,26.892471,321.260855,0.002715
3,Full_Feature_Set,XGB,0.705233,0.716430,32.024513,20.151809,0.196193,27.281481,306.197657,0.009546
4,Full_Feature_Set,LGBM,0.691547,0.706610,32.574280,19.979795,0.195348,27.749824,70.018718,0.021270
5,GEDI_Optical,RF,0.687679,0.704432,32.694967,21.184362,0.214934,27.852637,339.578268,0.033041
6,GEDI_Optical,CatBoost,0.697123,0.681999,33.913022,20.813428,0.208928,28.890290,223.378181,0.002450
7,GEDI_Optical,SVR,0.675477,0.680616,33.986717,19.166594,0.175473,28.953070,22.779824,0.033630
8,GEDI_Optical,LGBM,0.681445,0.677214,34.167252,21.160115,0.214024,29.106867,54.834548,0.018516
9,GEDI,CatBoost,0.679713,0.676236,34.218928,22.011233,0.224032,29.150889,112.746902,0.001848


In [ ]:
# ==========================================================
# SAVE SUMMARY TABLE
# ==========================================================

summary_path = os.path.join(
    TABLE_DIR,
    "Model_Comparison_Results.csv"
)

results_df.to_csv(
    summary_path,
    index=False
)

print(results_df)

print("\nSummary table saved.")

         Feature Set     Model  Best CV R2   Test R2       RMSE        MAE  \
0   Full_Feature_Set        RF    0.703216  0.731663  31.152472  18.717204   
1   Full_Feature_Set       SVR    0.677607  0.729638  31.269785  19.154242   
2   Full_Feature_Set  CatBoost    0.711262  0.724459  31.567871  19.753955   
3   Full_Feature_Set       XGB    0.705233  0.716430  32.024513  20.151809   
4   Full_Feature_Set      LGBM    0.691547  0.706610  32.574280  19.979795   
5       GEDI_Optical        RF    0.687679  0.704432  32.694967  21.184362   
6       GEDI_Optical  CatBoost    0.697123  0.681999  33.913022  20.813428   
7       GEDI_Optical       SVR    0.675477  0.680616  33.986717  19.166594   
8       GEDI_Optical      LGBM    0.681445  0.677214  34.167252  21.160115   
9               GEDI  CatBoost    0.679713  0.676236  34.218928  22.011233   
10              GEDI      LGBM    0.655672  0.675815  34.241196  21.567758   
11      GEDI_Optical       XGB    0.682603  0.674981  34.285206 